# 1. Introduction

### Exploratory Data Analysis and Model Implementation

This study focuses on detecting Distributed Denial-of-Service (DDoS) attacks using an unsupervised deep learning approach based on autoencoders. The CIC-DDoS2019 dataset is employed due to its diversity of modern attack scenarios and its flow-based feature representation. Prior to model development, an exploratory data analysis (EDA) is conducted to understand feature semantics, data quality, and statistical properties.

### Why CIC-DDoS2019 is used

This dataset contains recent network traffic data with binary labeling, distinguishing between benign traffic and DDoS attacks. In this configuration, all DDoS attack types are grouped under a single “DDoS” label to simplify the classification task and avoid ambiguity between specific attack variants.

The dataset is derived from real-world packet capture (PCAP) traffic and processed using CICFlowMeter-V3, which extracts flow-based statistical features from raw network packets. As a result, each record represents a labeled network flow rather than raw packet data.


License:
```
"DDoS Evaluation Dataset (CIC-DDoS2019).” DDoS 2019 | Datasets | Research | Canadian Insitute for Cybersecurity, 31 Oct. 2019, www.unb.ca/cic/datasets/ddos-2019.html.

Iman Sharafaldin, Arash Habibi Lashkari, Saqib Hakak, and Ali A. Ghorbani, "Developing Realistic Distributed Denial of Service (DDoS) Attack Dataset and Taxonomy", IEEE 53rd International Carnahan Conference on Security Technology, Chennai, India, 2019.
```

### Why unsupervised learning (autoencoder)

Autoencoders are well-suited for anomaly detection tasks because they can be trained primarily on unlabeled or normal traffic data. This reduces the dependency on large-scale labeled datasets, which are often costly and difficult to obtain in cybersecurity contexts.

Furthermore, once trained on benign behavior, the model learns a compressed representation of normal traffic patterns. Anomalies can then be identified based on reconstruction error using a predefined threshold, enabling the detection of previously unseen attack patterns.

# 2. Dataset Overview

In [16]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib as plt

df = pd.read_csv("./data/raw/CIC-DDos2019.csv")

# Several column have a blank space before the actual name
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.lower()

display(df.shape)
display(df.columns)
display(df.dtypes)

(225745, 85)

Index(['flow_id', 'source_ip', 'source_port', 'destination_ip',
       'destination_port', 'protocol', 'timestamp', 'flow_duration',
       'total_fwd_packets', 'total_backward_packets',
       'total_length_of_fwd_packets', 'total_length_of_bwd_packets',
       'fwd_packet_length_max', 'fwd_packet_length_min',
       'fwd_packet_length_mean', 'fwd_packet_length_std',
       'bwd_packet_length_max', 'bwd_packet_length_min',
       'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes/s',
       'flow_packets/s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max',
       'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std',
       'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean',
       'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags',
       'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets/s', 'bwd_packets/s',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
  

flow_id                 str
source_ip               str
source_port           int64
destination_ip          str
destination_port      int64
                     ...   
idle_mean           float64
idle_std            float64
idle_max              int64
idle_min              int64
label                   str
Length: 85, dtype: object

### Overall Structure

The dataset contains more than 225,000 rows, where each row represents a *network flow*. A flow is defined as a sequence of packets exchanged between two endpoints, which is aggregated using CICFlowMeter. The dataset is structured into 85 columns; however, not all features are equally informative. Therefore, the next step of the analysis focuses on identifying which features or groups of features are most relevant for model learning.

To achieve this, the features will be organized into meaningful categories, allowing for a more structured evaluation of their importance.

First, we categorize the features into distinct groups. Each group represents a different aspect of the network traffic that the autoencoder can learn from: *behavioral features*, which describe how a flow behaves; *identity features*, which characterize the communicating hosts; and *contextual features*, which provide information about the connection environment.

The objective of the model is to learn the typical behavior of benign network flows in order to identify those associated with malicious DDoS activity. To achieve this, *behavioral features* are expected to provide the most relevant information. *Some contextual features* may also be useful, as they help explain traffic patterns according to the nature of the connection. For example, a rate of 500 packets per second may be normal for a DNS flow, whereas it would be unusual for an SSH connection. Finally, *identity features* will be excluded from the analysis. Such features may introduce dataset-specific biases and encourage overfitting, since the model could rely on recurring identifiers (e.g., IP addresses, ports, and flow IDs) rather than learning the underlying behavioral patterns that distinguish benign traffic from DDoS attacks.

In [18]:
#We select the columns that correspond to identity features
identity_features = ['flow_id', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'timestamp']

try:
    df = df.drop(columns=identity_features)
except:
    print("Columns already dropped")

display(df.columns)
display(df.shape)

Columns already dropped


Index(['protocol', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'total_length_of_fwd_packets',
       'total_length_of_bwd_packets', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes/s', 'flow_packets/s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets/s', 'bwd_packets/s',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
       'syn_flag_count', 'r

(225745, 79)

### Near-constant features

Features with near-constant values provide little to no information gain, as they exhibit extremely low variance across the dataset. In an unsupervised setting such as autoencoders, these features can introduce noise in the optimization process without contributing meaningful structure for reconstruction or anomaly detection.
These datasets often carry infinite values, in order to that we will remove this by, first computing infinite values as NaN, and then droping every null value.

In [ ]:
#Features with less-equal 1 different value
const_features = df.columns[df.nunique() <= 1]

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

df = df.drop_duplicates()

try:
    df = df.drop(columns=const_features)
    print(f"{const_features} and null values dropped.")
except:
    print(f"{const_features} already dropped.")

display(df.columns)
display(df.shape)

Index(['bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'cwe_flag_count',
       'fwd_avg_bytes/bulk', 'fwd_avg_packets/bulk', 'fwd_avg_bulk_rate',
       'bwd_avg_bytes/bulk', 'bwd_avg_packets/bulk', 'bwd_avg_bulk_rate'],
      dtype='str') and null values dropped.


Index(['protocol', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'total_length_of_fwd_packets',
       'total_length_of_bwd_packets', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes/s', 'flow_packets/s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets/s', 'bwd_packets/s',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
       'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count',
      

(225711, 69)

### Redundant Features

Some features in this dataset do not contribute additional information for various reasons. For example, *fwd_header_length.1* is a known 'bug' of CICFlowMeter and is an exact duplicate of fwd_header_length. Other features, while not exact duplicates, are directly derived from existing variables and therefore contain highly redundant information. Since these values can be inferred from other features, explicitly including them may not provide additional predictive value and can unnecessarily increase the dimensionality of the dataset.e

##### Algebraic transformations
- *avg_fwd_segment_size* -> Exactly the same as *fwd_packet_length_mean* (same formula by a different name: **total_length_of_fwd_packets / total_fwd_packets**).
- *avg_bwd_segment_size* -> *bwd_packet_length_mean*.

##### Most connection flows consist of a single subflow
- *subflow_fwd_packets* -> *total_fwd_packets*.
- *subflow_fwd_bytes* -> *total_length_of_fwd_packets*.
- *subflow_bwd_packets* -> *total_backward_packets*.
- *subflow_bwd_bytes* -> *total_length_of_bwd_packets*.

##### Rate-Based Features
Features such as *flow_bytes/s*, *flow_packets/s*, *fwd_packets/s*, and *bwd_packets/s* are all derived from the general relationship:

**metric / flow_duration**

Given the underlying metric and the flow duration, these values can be directly computed. Therefore, they do not introduce additional information beyond what is already encoded in the original features.

In [27]:
redundant_features = ['fwd_header_length.1', 'avg_fwd_segment_size', 'avg_bwd_segment_size', 'subflow_fwd_packets', 'subflow_fwd_bytes', 'subflow_bwd_packets', 
                      'subflow_bwd_bytes', 'flow_bytes/s', 'flow_packets/s', 'fwd_packets/s', 'bwd_packets/s']

try:
    df = df.drop(columns=redundant_features)
except:
    print("Redundant features already dropped.")

display(df.columns)
display(df.shape)

Redundant features already dropped.


Index(['protocol', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'total_length_of_fwd_packets',
       'total_length_of_bwd_packets', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_iat_mean', 'flow_iat_std',
       'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean',
       'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total',
       'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min',
       'fwd_psh_flags', 'fwd_header_length', 'bwd_header_length',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
       'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count',
       'urg_flag_count', 'ece_flag_count', 'down/up_ratio',
       'average_packe

(225711, 59)

### Correlated Columns
Finally, a correlation analysis is performed to identify and remove highly correlated features from the dataset. A threshold of 0.95 is used; if any pair of features exhibits a correlation coefficient above this value, the corresponding variables are flagged for further analysis to determine whether one of them should be removed. This step helps reduce redundancy and mitigate multicollinearity in the feature space.

In [ ]:
threshold = 0.95

X = df.select_dtypes(include=["number"])

corr = X.corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "col1", "level_1": "col2", 0: "spearman"})
)

upper_pairs = pairs[pairs["spearman"] > threshold].sort_values("spearman", ascending=False)
pd.set_option("display.max_rows", None)
print(upper_pairs)

                             col1                    col2  spearman
1727            fwd_header_length     fwd_header_length.1  1.000000
1661                fwd_psh_flags          syn_flag_count  1.000000
2246               rst_flag_count          ece_flag_count  1.000000
2007            packet_length_std  packet_length_variance  0.999999
2952                  active_mean              active_max  0.999866
2953                  active_mean              active_min  0.999401
1360                bwd_iat_total             bwd_iat_max  0.999047
3069                   active_max              active_min  0.998938
3188                    idle_mean                idle_max  0.998591
1418                 bwd_iat_mean             bwd_iat_max  0.997070
1065                fwd_iat_total             fwd_iat_max  0.996355
1358                bwd_iat_total            bwd_iat_mean  0.996350
1123                 fwd_iat_mean             fwd_iat_max  0.995694
74                  flow_duration            flo

### Correlation Analysis
Looking at the correlation chart above, and taking into count the nature of the data we can find four different clusters of correlation types.

##### 1. Mathematic Correlation
- packet_length_variance = (packet_length_std)² -> The variance will be removed

##### 2. Packet Size
In flood attacks packet sizes converge to a same number, the different metrics are flattened. In this case we choose only one metric and remove the others.
We keep *fwd_packet_length_mean* and *bwd_packet_length_mean* and remove the max, min, std, etc.

##### 3. IAT
The IAT (Inter-Arrival Time) is defined as the time between the arrival of two consecutive packets. In automated environments this time could be highly regular, for this reason the flow_duration could be n_packets times a regular iat_time. We will keep flow_duration and fwd_iat_mean/bwd_iat_mean. And remove ['flow_iat_mean', 'flow_iat_total', 'fwd_iat_total', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min'].

##### 4. Active/Idle
As in the IAT, an automated attack could be so regular than the active and idle time are almost the same. We collapse this features in active and idle mean.

##### Special case: Flags
For protocol reasons *syn_flag_count* and *fwd_psh_flags* describe almost the same event. Equally with *rst_flag_count* and *ece_flag_count*. We keep the firs one of each pair.

In [ ]:
display(df[['fwd_psh_flags','syn_flag_count','rst_flag_count','ece_flag_count']].nunique())
display(df[['fwd_psh_flags','syn_flag_count','rst_flag_count','ece_flag_count']].value_counts())

to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

fwd_psh_flags     2
syn_flag_count    2
rst_flag_count    2
ece_flag_count    2
dtype: int64

fwd_psh_flags  syn_flag_count  rst_flag_count  ece_flag_count
0              0               0               0                 218194
1              1               0               0                   7490
0              0               1               1                     27
Name: count, dtype: int64